# Hawkes calibration and exact option-model entry point

The first section validates exponential and rough Hawkes event calibration on
a deterministic synthetic sample. The optional second section runs the exact
affine option calibration on the immutable GLD surface.


In [ ]:
import numpy as np
import pandas as pd

from Hawkes import (
    ExactHawkesCalibration, ExponentialHawkesCalibration,
    RoughHawkesCalibration,
)
from calibration_workflow import load_calibration_surface

SEED = 20260811
horizon = 250.0
tail_index, cutoff, branching = 0.45, 0.25, 0.55
alpha = branching * tail_index * cutoff ** tail_index
events = RoughHawkesCalibration.simulate(
    lambda0=0.35,
    alpha=alpha,
    tail_index=tail_index,
    cutoff=cutoff,
    horizon=horizon,
    seed=SEED,
)


In [ ]:
exponential_fit = ExponentialHawkesCalibration.fit(events, horizon)
rough_fit = RoughHawkesCalibration.fit(events, horizon)
pd.DataFrame([
    {"model": exponential_fit.model, "success": exponential_fit.success,
     "aic": exponential_fit.aic, "bic": exponential_fit.bic,
     **exponential_fit.params},
    {"model": rough_fit.model, "success": rough_fit.success,
     "aic": rough_fit.aic, "bic": rough_fit.bic,
     **rough_fit.params},
])


In [ ]:
RUN_EXACT_OPTION_CALIBRATION = False

if RUN_EXACT_OPTION_CALIBRATION:
    surface, spot = load_calibration_surface("Data")
    exact_result = ExactHawkesCalibration.calibrate_heston(
        surface,
        spot,
        maxiter=25,
        popsize=6,
        seed=SEED,
    )
    exact_parameters = ExactHawkesCalibration.unpack_heston_params(exact_result.x)
    display(exact_parameters)
